In [ ]:
!python --version
!nvidia-smi

In [ ]:
!pip install -q unsloth datasets

# Select Model from Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.2-3b-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,          # Auto-detect (bf16 if supported, else fp16)
    load_in_4bit=True,   # Enable 4-bit quantization
)

ModuleNotFoundError: No module named 'unsloth'

# Set Fine-Tuning Parameters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Load & Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)
print(dataset.column_names)

In [ ]:
from unsloth.chat_templates import to_sharegpt, standardize_sharegpt

dataset = to_sharegpt(
    dataset,
    merged_prompt="{instruction}",
    output_column_name="response",
    #conversation_extension=3,   
)

# Normalize the ShareGPT format
dataset = standardize_sharegpt(dataset)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3",
)

In [ ]:
def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in conversations
    ]
    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
print(dataset)

In [ ]:
print(dataset[1])

# Test Inference of Original Model

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "I want to cancel my order {{Order Number}}"
    }
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True
)

_ = model.generate(
    input_ids,
    streamer=text_streamer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
print(tokenizer.chat_template)

In [ ]:
print(
    tokenizer.decode(input_ids[0])
)

# Train on Dataset

In [ ]:
import trl
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=SFTConfig(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,

        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint="/kaggle/working/outputs/checkpoint-2250")

# Inference

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "I want to cancel my order {{Order Number}}"
    }
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True
)

_ = model.generate(
    input_ids,
    streamer=text_streamer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# Save

In [ ]:
model.save_pretrained("/kaggle/working/lora_model")
tokenizer.save_pretrained("/kaggle/working/lora_model")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/lora_model",
    "zip",
    "/kaggle/working/lora_model",
)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/gguf_model",
    "zip",
    "/kaggle/working/gguf_model",
)

In [ ]:
import os
from IPython.display import FileLink

# 1. Print current location to double-check
print("Current Working Directory:", os.getcwd())

# 2. List files to verify your target exists
print("Available files:", os.listdir('.'))

# 3. Enter ONLY the filename (e.g., 'model.pth' instead of '/kaggle/working/model.pth')
filename = 'gguf_model_gguf/llama-3.2-3b.Q4_K_M.gguf' 

if os.path.exists(filename):
    display(FileLink(filename))
else:
    print(f"Error: {filename} was not found in this folder!")

# Load Saved Model 

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/lora_model",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "I want to cancel my order 12345."
    }
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
)

_ = model.generate(
    input_ids,
    streamer=text_streamer,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
import unsloth
import transformers
import trl
import torch

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("unsloth:", unsloth.__version__)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,          
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)


In [ ]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/gguf_model_gguf/llama-3.2-3b.Q4_K_M.gguf') # Replace with your file name